In [1]:
!git clone https://github.com/hafsaShaban/flyrank_internship.git
%cd flyrank_internship

Cloning into 'flyrank_internship'...
remote: Enumerating objects: 111, done.
remote: Counting objects: 100% (111/111), done.
remote: Compressing objects: 100% (68/68), done.
remote: Total 111 (delta 28), reused 97 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (111/111), 1.82 MiB | 16.46 MiB/s, done.
Resolving deltas: 100% (28/28), done.
/content/flyrank_internship


In [2]:
!ls data/raw/


content_refresh_anonymized.csv


# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hafsaShaban/flyrank_internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why
## 1. My lane (or freestyle) and why

**Lane: Refresh / Content Opportunity Scoring** (Lane 2).

I'm picking this lane because it's the one lane where I already have concrete, repo-verified evidence that a learned model beats a simple rule, instead of just a plausible-sounding idea. The starter pipeline (`scripts/01-05`, run on the anonymized 30,000-row starter slice) already computed this comparison, and it's committed in `outputs/model_results.json`: a random forest scores **Precision@50 = 0.740** versus **0.240** for the transparent hand-written baseline rule. That means on a list of the top 50 pages flagged for review, the learned ranking gets roughly 37 genuinely relevant pages right versus about 12 for the fixed rule — using the exact same data. That gap is the reason this lane is worth pursuing: there's already proof of learnable signal beyond what a hand-tuned rule can capture, on real (anonymized) FlyRank data, not a hypothetical.

It also matches a real operational constraint I understand intuitively: a content review team has *limited capacity* — they can't manually re-check 30,000+ pages (or 519,606 in the full warehouse) every week, so *someone* has to decide which pages get looked at first. That's exactly the kind of decision-support ranking problem this lane is built for.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

## 2. The question: decision, action, cost of a wrong call

**Research question:** Given a page's last-90-day search and engagement signals, which pages should a content reviewer look at first this week — and why?

**Unit of analysis:** one row = **one content page** (`content_id`), aggregated over a trailing 90-day window. In the starter dataset this is one row per pseudonymized page across 32 clients; in the full warehouse it's the same grain via `dim_content` joined to `fact_content_daily_performance`.

**The decision this improves:** which pages a human content reviewer opens first, out of a backlog far larger than their weekly review capacity.

**Who acts on it:** a content/SEO reviewer (at FlyRank or a client-facing strategist) working through a ranked queue, not an automated system — this is decision support, not an autonomous action.

**The output:** a ranked list of pages, each with a `final_refresh_score`, a suggested action (e.g. refresh, expand, protect, monitor), and a reason code the reviewer can actually inspect (e.g. `stale_visible_page`, `declining_with_demand`, `low_ctr_visible_page`) rather than a black-box number.

**Cost of a wrong call, in both directions:**
- **False positive** (a page is ranked high but wasn't actually a problem): wastes a reviewer's limited weekly time — time that could have gone to a page that genuinely needed it. With capacity for maybe 20-50 reviews a week, every wasted slot is a real opportunity cost.
- **False negative** (a genuinely declining, high-value page never surfaces in the queue): the page keeps losing visibility/traffic silently, and the client only finds out once the damage has compounded — the more expensive failure mode, since demand (`impressions_90d`) is already there and being lost.

**Why data/ML can help at all (not just 'train a model'):** the starter pipeline's own result is the evidence — a transparent baseline rule already exists (`baseline_refresh_score`), and a simple learned model measurably reorders the queue better than that rule on held-out clients (client-holdout split, not random split, so it's not just memorizing individual clients). That's a real, checkable improvement in *how well limited review capacity gets spent* — which is the actual decision, not an abstract accuracy number.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [4]:
# Real number 3: the committed model-vs-baseline comparison already in the repo
with open('outputs/model_report.md') as f:
    report = f.read()
print(report)

# FlyRank Refresh Opportunity Model Report

This report is generated from the bundled anonymized starter dataset (`data/raw/content_refresh_anonymized.csv`).
The model ranks existing content for refresh review. It does not use titles, URLs, client names, domains, or keywords.

## Data

- Rows scored: 30,000
- Declining-label rows: 16,262
- Declining-label rate: 0.542
- Split strategy used for validation: client_holdout
- Target: `is_declining_label`

## Model Comparison

Best model: `random_forest` selected by `precision_at_50`.

| Model | ROC AUC | Avg precision | Precision@50 | Recall | F1 |
|---|---:|---:|---:|---:|---:|
| decision_tree | 0.742 | 0.575 | 0.540 | 0.716 | 0.634 |
| logistic_regression | 0.700 | 0.522 | 0.400 | 0.567 | 0.566 |
| random_forest | 0.750 | 0.618 | 0.740 | 0.744 | 0.640 |
| baseline_rules | 0.627 | 0.468 | 0.240 | - | - |

## Final Queue

- High-confidence items: 3,605
- Medium-confidence items: 11,395
- Low-confidence items: 15,000
- `monitor` items: 13,09

## 4. Careful words: what I can and can't claim

## 4. Careful words: what I can and can't claim

**What this work CAN say, if the results hold up through validation:**
- That certain *observed* signals (impressions, position, CTR, freshness, engagement) are *associated* with a page currently trending down, based on this anonymized 90-day slice.
- That a learned ranking is *directionally* better than the fixed baseline rule at ordering pages by review priority, measured with Precision@K on a client-holdout split.
- That the resulting queue is *decision-support*: a ranked starting point for a human reviewer with reason codes they can check — not an automated verdict.

**What this work CANNOT say, no matter how good the numbers look:**
- It cannot claim refreshing a page **causes** recovery — correlation between 'flagged' and 'later improved' is not proof; that would need a real experiment (e.g. A/B refreshing matched pages), which this dataset doesn't support.
- It cannot claim to have discovered or predicted anything about **Google's ranking algorithm** — only observed search/engagement outcomes, never the algorithm itself.
- The current label, `is_declining_label` (`trend_direction == 'down'`), is a **proxy defined from the current window**, not a true future outcome — so early results describe 'pages that already look like they're declining right now,' not 'pages that will decline next month.' A stronger version of this lane (weeks 3+, using the warehouse release) would redefine the label as a genuine future-window outcome to fix this.
- Any tier-level number (e.g. median CTR in `top_3` position tier) is unreliable without also stating the volume floor behind it — some tiers here have very low median impression counts, where a single click swings the percentage several points.
- Nothing here should be read as proof about a specific client, domain, or page — all IDs are pseudonymous and this analysis stays observational and aggregate.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.